# Variant 1 — gauge-combo transformer, **L=4, (h_x, h_z) = (0.2, 0.2)** (minSR + W&B)

First experiment where exact A_v symmetry does **NOT** hold (h_z anticommutes with A_v = XXXX).
Trains the **Variant 1** ansatz (branch `variant3-plaquette-transformer`):

```
σ → odd embed σ_e·w_orient → n1 × χ blocks (gated full attention, IDENTITY at init:
zero-init W_O & ffn1; tanh, bias-free, pre-RMSNorm) → fixed Wilson ∏₄ tanh(·) per channel
→ Dense(C→d) → n2 × Variant-3 encoder blocks → sum-pool → RMSNorm → Dense_K → Σ log-cosh
```

At init this is exactly Variant 3 (A_v gate prints 0); training breaks A_v by the learned
dressing χ — that's how h_z is captured. Arms: `v1` (gated), `v1f` (frozen α_h — factored
ablation), `cnn` (the paper's Combo-small baseline, same W&B project for side-by-side curves).
**ED runs FIRST** so every training cell prints against ground truth.

## Before you start
- Runtime → Change runtime type → **GPU**.
- GitHub token (repo read access) + W&B API key (wandb.ai/authorize, optional).


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4 wandb
print("install done")

In [ ]:
# CRITICAL: stop THIS notebook kernel from preallocating 75% of the GPU.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # must run BEFORE importing jax

import jax, netket, flax
print("jax", jax.__version__, "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "NO GPU -- fix the runtime type first"
try:
    from netket.experimental.driver import VMC_SRt as _MinSR
except ImportError:
    from netket.driver import VMC_SR as _MinSR
print("minSR driver OK:", _MinSR.__name__, "from", _MinSR.__module__)

## 3. Mount Google Drive  *(resume point after any runtime restart)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive

In [ ]:
import os, getpass, subprocess, shutil

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"
BRANCH     = "variant3-plaquette-transformer"
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC-variant1")    # dedicated dir for THIS experiment
os.makedirs(DRIVE_ROOT, exist_ok=True)

def git(args, secret=None, check=True):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if check and r.returncode != 0:
        clean = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
        print("GIT FAILED:", clean(" ".join(args)))
        print(clean(r.stderr).strip())
        raise RuntimeError("git command failed (see the GIT FAILED output above)")
    return r

need_clone = not os.path.isdir(os.path.join(REPO_DIR, ".git"))
if not need_clone and git(["-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"], check=False).returncode != 0:
    print("existing repo looks broken -> removing and re-cloning")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    need_clone = True

tok = getpass.getpass("GitHub token (repo read access): ")
url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
try:
    if need_clone:
        print("cloning", BRANCH, "->", REPO_DIR)
        git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], secret=tok)
        git(["-C", REPO_DIR, "remote", "set-url", "origin",
             "https://github.com/SanzharBissenali/2D-TC.git"])
    else:
        print("repo present; best-effort update of", BRANCH)
        for lk in (".git/index.lock", ".git/shallow.lock", ".git/HEAD.lock"):
            lp = os.path.join(REPO_DIR, lk)
            if os.path.exists(lp):
                os.remove(lp); print("removed stale", lk)
        try:
            git(["-C", REPO_DIR, "fetch", url, "--depth", "1", f"{BRANCH}:refs/remotes/origin/{BRANCH}"], secret=tok)
            git(["-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
        except Exception as e:
            print("update skipped (using existing checkout):", str(e)[:150])
finally:
    del tok, url

os.chdir(REPO_DIR)
head = git(["-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert os.path.exists("model/gauge_combo_transformer.py"), "Variant-1 code missing -- pull the latest branch"
print("OK: Variant-1 code present.")

## 5. Weights & Biases login *(optional)*
Same project **`2d-tc-transformer`** as Variant 3, group `variant1-L4` — v1/v1f/cnn curves side by side.

In [ ]:
import os, getpass, wandb
os.environ["WANDB_MODE"] = "online"
wandb.login(key=getpass.getpass("W&B API key (wandb.ai/authorize): "))
print("W&B login OK")

## 6. Hyperparameters
Ω defaults = the Variant-3 sweep winner (d16/nl4/h4/gelu, lr 0.01, ds 1e-4). χ starts minimal
(1 block, C=8, 2 heads) per the spec ("accuracy scales with invariant-block depth, not the
non-invariant block"). The auto-TAG covers χ and Ω knobs; CNN runs get their own minimal tag.

In [ ]:
import os, json

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
EDDIR  = os.path.join(REPO, "results", "ed")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(EDDIR, exist_ok=True)

HP = dict(
    # --- physics: the A_v-breaking benchmark point ---
    LX          = 4,
    HX          = 0.20,
    HZ          = 0.20,
    # --- training ---
    OPTIMIZER   = "minsr",
    N_STEPS     = 350,
    LR          = 0.01,
    DT          = 0.01,
    DIAG_SHIFT  = 1e-4,
    LR_SCHEDULE = "const",
    N_SAMPLES   = 8192,
    CHUNK_SIZE  = 1024,
    SEED        = 0,
    # --- chi (edge cleaning block) ---
    CHI_LAYERS  = 1,
    CHI_DMODEL  = 8,           # C channels per edge token
    CHI_HEADS   = 2,
    # --- Omega (Variant-3 backbone; sweep-winner defaults) ---
    TF_LAYERS   = 4,
    TF_DMODEL   = 16,
    TF_HEADS    = 4,
    TF_FFN_MULT = 2,
    TF_ACT      = "gelu",
    TF_REMAT    = False,
    # --- bookkeeping ---
    WANDB       = True,
    WANDB_PROJECT = "2d-tc-transformer",
    WANDB_GROUP   = "variant1-L4",
)

def _tag(hp, arm="v1"):
    if arm == "cnn":
        return f"_dt{hp['DT']:g}_ds{hp['DIAG_SHIFT']:g}"
    d_head = hp["TF_DMODEL"] // hp["TF_HEADS"]
    t = (f"_c{hp['CHI_DMODEL']}x{hp['CHI_LAYERS']}"
         f"_nl{hp['TF_LAYERS']}_lr{hp['LR']:g}_ds{hp['DIAG_SHIFT']:g}"
         f"_d{hp['TF_DMODEL']}dh{d_head}")
    if hp["TF_FFN_MULT"] != 2:  t += f"_fm{hp['TF_FFN_MULT']}"
    if hp["TF_ACT"] != "gelu":  t += f"_{hp['TF_ACT']}"
    if hp["LR_SCHEDULE"] != "const":  t += f"_{hp['LR_SCHEDULE']}"
    if hp["N_SAMPLES"] != 8192:  t += f"_ns{hp['N_SAMPLES']}"
    if hp["SEED"] != 0:  t += f"_s{hp['SEED']}"
    return t

def _jobid(arm, hp=None):
    hp = hp or HP
    return f"L{hp['LX']}_hx{hp['HX']:.2f}_hz{hp['HZ']:.2f}_{arm}{_tag(hp, arm)}"

def _base(arm, hp=None):
    return os.path.join(OUTDIR, f"G-equiv_1_{_jobid(arm, hp)}")

print("jobid preview v1 :", _jobid("v1"))
print("jobid preview cnn:", _jobid("cnn"))

## 7. ED ground truth — runs FIRST *(needs a High-RAM runtime!)*
The L=4 sparse Hamiltonian peaks ~10 GB of **system RAM** (CPU job — the GPU doesn't help);
a standard Colab runtime (~12.7 GB) gets OOM-killed (`exit -9`). Use **Runtime → Change runtime
type → High-RAM** (or the A100 runtime, 83 GB). `--no-observables`: only `E0` is needed for the
rel-err, and it avoids dying in the post-diagonalization observable pass (the JSON is written
only at the very end). Skips if the JSON already exists.

In [ ]:
import subprocess, psutil

def stream(cmd):
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(["bash", "-lc", cmd], env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()

ram_gb = psutil.virtual_memory().total / 1e9
print(f"system RAM: {ram_gb:.1f} GB")
assert ram_gb > 20, ("Not enough system RAM for L=4 ED (~10 GB peak; standard Colab gets "
                     "OOM-killed). Runtime -> Change runtime type -> High-RAM, then re-run.")

ED_PATH = os.path.join(EDDIR, f"ed_L{HP['LX']}_hx{HP['HX']:.2f}_hz{HP['HZ']:.2f}.json")
if os.path.exists(ED_PATH):
    print("ED present:", ED_PATH)
else:
    rc = stream(f"cd {REPO} && python -u -m exact.lanczos_ed "
                f"--Lx {HP['LX']} --hx {HP['HX']} --hz {HP['HZ']} --k 4 --no-observables "
                f"--out {ED_PATH}")
    print("ED exit", rc)
    assert rc == 0, f"ED failed (exit {rc}; -9 = OOM-killed -> use a High-RAM runtime)"
E0 = float(json.load(open(ED_PATH))["E0"])
print(f"ED ground state: E0 = {E0:.8f}")

## 8. Training driver — arms: `v1` (gated), `v1f` (frozen α), `cnn` (baseline)
The A_v gate line for v1/v1f must print `0.00e+00` at init (identity χ). The CNN arm uses its
validated native setup (dense-QGT tdvp, diag_shift 6e-5) — note the optimizer differs from the
minSR transformer arms, so compare converged quality, not step-for-step speed.

In [ ]:
def run_arm(arm="v1", force=False, **overrides):
    """Train one arm at (hx,hz), STREAMING per-step progress live. Overrides merge onto HP."""
    hp = {**HP, **overrides}
    jobid = _jobid(arm, hp)
    base = _base(arm, hp)
    done_marker = base + ("_attn.json" if arm != "cnn" else ".mpack")
    if not force and os.path.exists(base + ".mpack") and os.path.exists(done_marker):
        print(f"== skip {jobid} (already complete; force=True to re-run) =="); return

    wb = (f"--wandb --wandb_project {hp['WANDB_PROJECT']} --wandb_group {hp['WANDB_GROUP']}"
          if hp["WANDB"] else "")
    if arm == "cnn":
        arch = "--symmetric_block cnn"
        opt  = f"--optimizer tdvp"
    else:
        content = "--tf_content" if arm != "v1f" else "--no-tf_content"
        remat   = " --tf_remat" if hp.get("TF_REMAT") else ""
        arch = (f"--symmetric_block variant1 "
                f"--tf1_layers {hp['CHI_LAYERS']} --tf1_dmodel {hp['CHI_DMODEL']} "
                f"--tf1_heads {hp['CHI_HEADS']} "
                f"--tf_layers {hp['TF_LAYERS']} --tf_dmodel {hp['TF_DMODEL']} "
                f"--tf_heads {hp['TF_HEADS']} --tf_ffn_mult {hp['TF_FFN_MULT']} "
                f"--tf_activation {hp['TF_ACT']} {content}{remat}")
        opt  = f"--optimizer {hp['OPTIMIZER']} --lr {hp['LR']}"
    cmd = (
        f"cd {OUTDIR} && python -u {REPO}/main.py "
        f"--outindex 1 --jobid {jobid} "
        f"--Lx {hp['LX']} --hx {hp['HX']} --hy 0.0 --hz {hp['HZ']} "
        f"{opt} --n_steps {hp['N_STEPS']} "
        f"--dt {hp['DT']} --diag_shift {hp['DIAG_SHIFT']} --lr_schedule {hp['LR_SCHEDULE']} "
        f"{arch} "
        f"--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
        f"--n_samples_fin {hp['N_SAMPLES']} --chunk_size {hp['CHUNK_SIZE']} "
        f"--use_custom_sampler --seed {hp['SEED']} {wb}"
    )
    print(f"RUN {jobid}  ({'tdvp' if arm=='cnn' else hp['OPTIMIZER']}, {hp['N_STEPS']} steps)", flush=True)
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1",
               XLA_PYTHON_CLIENT_MEM_FRACTION="0.90")
    proc = subprocess.Popen(["bash", "-lc", cmd], env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    print(f"===== {jobid} exit {proc.wait()} =====", flush=True)

## 9. Train

In [ ]:
run_arm("v1")                                 # gated full attention chi + Omega

In [ ]:
run_arm("v1f")                                # ablation: frozen alpha_h=0 everywhere

In [ ]:
run_arm("cnn", DIAG_SHIFT=6e-5)               # paper baseline (Combo-small, tdvp)

### Overnight sweep *(optional — sequential, resumable, each setting its own run)*

In [ ]:
SWEEP = [
    dict(CHI_LAYERS=2),                            # deeper cleaning block
    dict(CHI_DMODEL=16, CHI_HEADS=4),              # wider cleaning block
    dict(TF_DMODEL=32, TF_HEADS=4),                # wider Omega (the L=4 escalation knob)
    dict(LR_SCHEDULE="cosine", N_STEPS=700),       # tail-tightening schedule
]
for i, over in enumerate(SWEEP):
    print(f"\n########## sweep {i+1}/{len(SWEEP)}: {over} ##########", flush=True)
    run_arm("v1", **over)
print("\nSWEEP DONE")

## 10. Learning curves, ED comparison, α_h gates
Watch the χ gates (`chi0…`) separately from Ω's (`block0…`): with h_z≠0 the χ gates are the
first place content routing should actually switch on (the syndrome-matching claim).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

ARMS = ["v1", "v1f", "cnn"]

def load(arm):
    try:
        return json.load(open(_base(arm) + ".json"))
    except FileNotFoundError:
        return None

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for arm in ARMS:
    d = load(arm)
    if d is None:
        print(f"({arm}: no run yet)"); continue
    E = np.array([complex(x).real for x in d["energy"]])
    V = np.array([abs(complex(x)) for x in d["Vscore"]])
    rel = abs(E[-1] - E0) / abs(E0)
    print(f"{arm:4s}: E = {E[-1]:.8f}   rel-err vs ED = {rel:.3e}"
          + ("   <= 1e-4 TARGET MET" if rel <= 1e-4 else ""))
    ax[0].plot(E, label=f"{arm} ({rel:.1e})")
    ax[1].semilogy(V, label=arm)
ax[0].axhline(E0, color="k", ls="--", lw=1, label="ED")
ax[0].set(xlabel="step", ylabel="E", title=f"L={HP['LX']}, (hx,hz)=({HP['HX']},{HP['HZ']})")
ax[0].legend()
ax[1].set(xlabel="step", ylabel="V-score", title="V-score (lower = better)"); ax[1].legend()
plt.tight_layout(); plt.show()

for arm in ("v1", "v1f"):
    p = _base(arm) + "_attn.json"
    if os.path.exists(p):
        alphas = json.load(open(p)).get("alpha", {})
        gates = {k: v for k, v in alphas.items() if np.size(v) <= 16}
        print(f"\n{arm} content gates alpha_h (chi* = cleaning block, block* = Omega):")
        for blk, vals in sorted(gates.items()):
            print(f"  {blk:>10s}: " + "  ".join(f"{float(v):+.4f}" for v in np.ravel(vals)))